## Step 6 - OCR extraction (EasyOCR), one JSON per video

Reads the text burned into every keyframe with **EasyOCR** (`vi` + `en`) and
writes **one JSON file per video** so the dataset is trivial to extend: when new
videos arrive, re-run and only the missing videos are processed.

```
OCR/<video_name>.json     # { "<frame_id>": "text", ... }  e.g. OCR/L21_V001.json
```

`frame_id = <video_name>_<idx>` (e.g. `L21_V001_000080`) - the same join key the
backend's `ocr_service` uses, so the per-video files merge straight into
`data/ocr/ocr_data.json` (last cell) with no remapping.

**Resumable on two levels**
- *per video* - a video whose `OCR/<video_name>.json` already covers all its
  frames is skipped entirely;
- *per frame* - within a video, frames already in that JSON are not re-read, and
  the file is checkpointed every `CHECKPOINT_EVERY` frames.

Input is the keyframe dataset from Step 3 (`Keyframes/<batch>/<video_name>/<idx>.webp`).
Use a **GPU** session for speed.

> Why EasyOCR: single `pip install`, stable API, no numpy/paddle version clashes
> and no kernel restart - it just runs on Kaggle's stock environment.

In [ ]:
# No -U: install easyocr + only its missing deps, without force-upgrading
# numba etc. (the "ERROR: pip's dependency resolver" lines about dask-cuda /
# cuml / cudf are RAPIDS warnings, unrelated to OCR and safe to ignore).
!pip -q install easyocr

In [ ]:
import os
import json
from pathlib import Path

from tqdm.auto import tqdm
import easyocr

In [ ]:
# Step 3 keyframes (nested: Keyframes/<batch>/<video_name>/<idx>.webp)
KEYFRAMES_DIR    = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR       = './OCR'              # one <video_name>.json per video
LANGS            = ['vi', 'en']         # Vietnamese + English (both latin -> compatible)
IMAGE_EXTS       = ('.jpg', '.jpeg', '.png', '.webp')
CHECKPOINT_EVERY = 16                   # save the video's JSON every N frames

# readtext tuning for downscaled 640p broadcast frames where headline/banner text
# on busy backgrounds is otherwise missed at the defaults.
READ_KW = dict(
    detail=1,
    paragraph=False,
    mag_ratio=1.5,         # default 1.0 -> upscale small frames before detection
    text_threshold=0.6,    # default 0.7 -> accept slightly fainter text
    low_text=0.3,          # default 0.4 -> grow boxes to catch full words
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# One reader, shared across all videos (loads weights once).
# Kaggle randomly assigns P100 (sm_60) or T4 (sm_75); the stock PyTorch only
# supports sm_70+, so on a P100 GPU init raises "no kernel image". Best fix:
# set Accelerator = "GPU T4 x2" in the notebook settings. Falls back to CPU here
# so the run still works (slower) if you're stuck on a P100.
try:
    reader = easyocr.Reader(LANGS, gpu=True)
    print('easyocr ready (GPU), langs =', LANGS)
except Exception as e:
    print(f'GPU init failed ({type(e).__name__}) -> using CPU. '
          'For speed switch Accelerator to "GPU T4 x2" (P100 is sm_60, '
          'unsupported by this PyTorch).')
    reader = easyocr.Reader(LANGS, gpu=False)
    print('easyocr ready (CPU), langs =', LANGS)

In [ ]:
def _texts_from(result):
    """Pull recognised strings out of an EasyOCR result.

    reader.readtext(img, detail=1) -> [ [box, text, conf], ... ].
    """
    return [r[1] for r in (result or []) if isinstance(r, (list, tuple)) and len(r) >= 2]

def ocr_image(path):
    return ' '.join(_texts_from(reader.readtext(str(path), **READ_KW))).strip()

### Debug a single hard frame (optional)

Run this on a frame that came back empty to see what EasyOCR detects + its
confidence. Nothing printed -> loosen `text_threshold` / `low_text` or raise
`mag_ratio` in `READ_KW`; boxes found but text wrong -> a recogniser/font limit
(decorative logos are fine to miss).

In [ ]:
dbg_path = '/kaggle/input/aic-keyframes-webp/Keyframes/L21/L21_V001/000080.webp'  # <- a hard frame
res = reader.readtext(dbg_path, **READ_KW)
for box, text, conf in res:
    print(f'{conf:.2f}  {text}')
print('--> joined:', repr(' '.join(_texts_from(res)).strip()))

In [ ]:
def out_path(video_name):
    return Path(OUTPUT_DIR) / f'{video_name}.json'

def load_video_ocr(video_name):
    p = out_path(video_name)
    if p.exists():
        with p.open(encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_video_ocr(video_name, data):
    with out_path(video_name).open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
# Group every keyframe by its video folder: {video_name: [(frame_id, path), ...]}
root = Path(KEYFRAMES_DIR)
videos = {}
for img in root.rglob('*'):
    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
        video_name = img.parent.name              # e.g. L21_V001
        frame_id = f'{video_name}_{img.stem}'     # img.stem is the zero-padded idx
        videos.setdefault(video_name, []).append((frame_id, img))

for v in videos:
    videos[v].sort(key=lambda t: t[0])
print('videos found:', len(videos),
      '| frames:', sum(len(v) for v in videos.values()))

In [ ]:
for video_name in tqdm(sorted(videos), desc='videos'):
    frames = videos[video_name]
    data = load_video_ocr(video_name)

    todo = [(fid, p) for fid, p in frames if fid not in data]
    if not todo:
        continue                                   # video already complete -> skip

    for n, (fid, p) in enumerate(todo, 1):
        try:
            data[fid] = ocr_image(p)
        except Exception as e:
            print('WARN', p, '->', e)
            data[fid] = ''
        if n % CHECKPOINT_EVERY == 0:
            save_video_ocr(video_name, data)       # periodic checkpoint
    save_video_ocr(video_name, data)               # flush the rest

print('done. per-video JSON in', OUTPUT_DIR)

### Merge into the backend's single `ocr_data.json` (optional)

The backend's `ocr_service` reads one flat dict. The per-video files merge into it
by a plain `dict.update`, so you can rebuild `ocr_data.json` any time without
re-running OCR - and adding a new video only means dropping in its JSON and
re-merging. Copy the result to `backend/data/ocr/ocr_data.json`.

In [ ]:
merged = {}
for jf in sorted(Path(OUTPUT_DIR).glob('*.json')):
    with jf.open(encoding='utf-8') as f:
        merged.update(json.load(f))

with open('ocr_data.json', 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)
print('merged entries:', len(merged), '-> ocr_data.json')